# RF-DETR - Object Detection on Geospatial Imagery

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/rfdetr_detection.ipynb)

This notebook demonstrates how to use [RF-DETR](https://github.com/roboflow/rf-detr) integration in GeoAI for object detection on geospatial imagery.

RF-DETR is Roboflow's state-of-the-art real-time object detection model built on DINOv2 and DETR. Key features:

- **High accuracy**: Achieves top performance on COCO benchmark and aerial imagery tasks
- **Multiple model sizes**: From Nano (384px, fastest) to 2XLarge (880px, most accurate)
- **Instance segmentation**: Segmentation variants available for mask prediction
- **Georeferenced output**: Tiled inference on GeoTIFF with results as GeoDataFrame

In this notebook, we will:

1. Fine-tune RF-DETR on the [NWPU-VHR-10](https://github.com/chaozhong2010/VHR-10_dataset_coco) aerial object detection dataset
2. Use the trained model to detect vehicles in high-resolution aerial imagery
3. Visualize georeferenced detection results on an interactive map

## Installation

Uncomment the following line to install the required packages.

In [ ]:
# %pip install -U "geoai-py[rfdetr]" "rfdetr[train]"

## Import Libraries

In [ ]:
import geoai

## List Available Models

RF-DETR provides 11 model variants: 5 detection models and 6 segmentation models. Models are automatically downloaded and cached on first use.

In [ ]:
from geoai import list_rfdetr_models, RFDETR_MODELS

for name, desc in list_rfdetr_models().items():
    resolution = RFDETR_MODELS[name]["resolution"]
    print(f"  {name:15s} [{resolution:4d}px] {desc}")

## Prepare Training Dataset

The NWPU-VHR-10 dataset contains 650 annotated VHR (Very High Resolution) remote sensing images with 10 object classes: airplane, ship, storage_tank, baseball_diamond, tennis_court, basketball_court, ground_track_field, harbor, bridge, and vehicle.

The `prepare_nwpu_for_rfdetr()` function downloads the dataset and organizes it into the COCO format directory structure expected by RF-DETR.

In [ ]:
result = geoai.prepare_nwpu_for_rfdetr(output_dir="nwpu-rfdetr")
print(f"Dataset directory: {result['dataset_dir']}")
print(f"Classes: {result['class_names']}")
print(f"Training images: {result['train_images']}")
print(f"Validation images: {result['val_images']}")

## Train RF-DETR

Fine-tune RF-DETR Base model on the NWPU-VHR-10 dataset. Training starts from COCO pretrained weights and adapts to the 10 aerial object classes. Adjust `epochs` and `batch_size` based on your GPU memory.

In [ ]:
output_dir = geoai.rfdetr_train(
    dataset_dir=result["dataset_dir"],
    model_variant="base",
    epochs=10,
    batch_size=4,
    output_dir="rfdetr-nwpu-output",
)
print(f"Training complete! Output: {output_dir}")

## Plot Training Metrics

Visualize the training loss, validation mAP, F1/precision/recall, and per-class AP curves.

In [ ]:
metrics_df = geoai.plot_rfdetr_metrics(output_dir)
metrics_df

## Download Sample Data for Inference

Download a high-resolution aerial image (7 cm/px) to test our trained model.

In [ ]:
url = "https://data.source.coop/opengeos/geoai/cars-7cm.tif"
image_path = geoai.download_file(url)

## Visualize the Input Image

In [ ]:
geoai.view_raster(image_path, backend="ipyleaflet")

## Detect Objects with Fine-Tuned Model

Run RF-DETR detection on the aerial image using our fine-tuned model. The function uses tiled inference with a sliding window, merges overlapping detections with NMS, and returns a georeferenced GeoDataFrame.

In [ ]:
weights_path = "rfdetr-nwpu-output/checkpoint_best_total.pth"
class_names = result["class_names"]

gdf = geoai.rfdetr_detect(
    image_path,
    model_variant="base",
    pretrain_weights=weights_path,
    confidence_threshold=0.3,
    class_names=class_names,
)
print(f"Detected {len(gdf)} objects")
gdf.head(10)

### Detection Statistics

Summarize detection results by class.

In [ ]:
print("Detections by class:")
print(gdf["class_name"].value_counts().to_string())
print(
    f"\nConfidence range: {gdf['confidence'].min():.3f} - {gdf['confidence'].max():.3f}"
)
print(f"Mean confidence: {gdf['confidence'].mean():.3f}")

### Visualize All Detections

Overlay all detection results on the original image.

In [ ]:
geoai.view_vector_interactive(gdf, tiles=image_path)

### Filter by Class

Filter detections to show only vehicles.

In [ ]:
vehicles = gdf[gdf["class_name"] == "vehicle"]
print(f"Vehicles detected: {len(vehicles)}")
geoai.view_vector_interactive(vehicles, tiles=image_path)

### Save Detection Results

Export detections to GeoJSON.

In [ ]:
gdf.to_file("rfdetr_detections.geojson", driver="GeoJSON")
print("Saved detections to rfdetr_detections.geojson")

## HuggingFace Hub Integration

After training, you can push your model to HuggingFace Hub for sharing and deployment, and later download it for inference.

```python
# Push the trained model to HuggingFace Hub
url = geoai.push_rfdetr_to_hub(
    model_path="rfdetr-nwpu-output/checkpoint_best_total.pth",
    repo_id="username/rfdetr-nwpu-vhr10",
    model_variant="base",
    class_names=result["class_names"],
)
print(f"Model uploaded to: {url}")

# Later, run detection using a model from HuggingFace Hub
gdf = geoai.rfdetr_detect_from_hub(
    input_path="image.tif",
    repo_id="username/rfdetr-nwpu-vhr10",
    confidence_threshold=0.3,
)
```